In [ ]:
from STopover import STopover_imageST
import scanpy as sc
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

import sys, os
sys.path.append(os.path.join("..", "3.Segmentation of tumor cells and vessel area in spatial transcriptomic data"))
from processing import find_cell_areas

In [ ]:
directory = '../data/h5ad_batch_55um'
file_list = os.listdir(directory)
file_list.sort()

In [ ]:
directory = '../data/h5ad_batch/'  # Change this to your target directory path.
file_list = os.listdir(directory)
file_list.sort()

adata_list = []
for file_name in file_list:
    adata = sc.read_h5ad(f'{directory}{file_name}')     
    adata.layers['counts'] = adata.X
    sc.pp.normalize_total(adata, target_sum = 1e3)
    sc.pp.log1p(adata)
    adata.layers['log1p'] = adata.X
    adata_list.append(adata)


# Update the loop to ensure that any COO matrix is converted before further processing
sample_list = [file.split('.')[0] for file in file_list]
adata_list_fin = []

for i, adata in enumerate(adata_list):
    batch_list = adata.obs['batch'].unique().tolist()
    sample_id = sample_list[i]
    for batch in batch_list:
        adata_ = adata[adata.obs['batch'] == batch].copy()
        
        # Process cell areas (this function should work as intended)
        find_cell_areas(adata=adata_)  # Default. cancer_epithelial_filtered
        
        # Convert adata_.X to CSR if it is in COO format to avoid warning issues
        if hasattr(adata_.X, 'format') and adata_.X.format == 'coo':
            adata_.X = adata_.X.tocsr()
        
        adata_list_fin.append(adata_)

In [ ]:
CTSgene = ["CTSB", "CTSL", "CTSA", "CTSC", "CTSS", "CTSK", "CTSH"]
her_family_gene = ["EGFR", "ERBB3", "ERBB4"]
list_1 = her_family_gene + CTSgene
list_2 = ['ERBB2']*len(list_1)
feature_pairs = pd.DataFrame([list_1, list_2])
feature_pairs = feature_pairs.T

In [ ]:
adata_cc = []
for adata in adata_list_fin:
    adata_ = adata[adata.obs['is_Cancer_Epithelial_filtered']].copy()
    sp_adata = STopover_imageST(sp_adata=adata_,
                                min_size=20, fwhm=2.5, thres_per=30, 
                                save_path='STopover') 
    sp_adata.topological_similarity(feat_pairs=feature_pairs, J_result_name='result',
                                    num_workers = 20)
    adata_cc.append(sp_adata)

In [ ]:
result_df = []
for adata in adata_cc:
    for gene in list_1:
        if f'Comb_CC_{gene}' not in adata.obs.columns:
            continue

        a = sum(adata.obs['Comb_CC_ERBB2'] != 0)
        b = sum((adata.obs['Comb_CC_ERBB2'] != 0) & (adata.obs[f'Comb_CC_{gene}'] != 0))

        if a == 0:
            prop = 0
        else:
            prop = b / len(adata)

        batch = adata.obs['batch'].unique().tolist()[0]
        results = [batch, gene, prop]
        result_df.append(results)



In [ ]:
result_df = pd.DataFrame(result_df, columns = ['batch', 'gene', 'prop'])
result_df

In [ ]:
results_df_raw = pd.read_csv('../data/results_df_spatial_vessel.csv', index_col = 0)

In [ ]:
result_df_CTSB = result_df[result_df['gene'] == 'CTSB'].copy()
result_df_CTSB = result_df_CTSB.reset_index(drop = True)
result_df_CTSB = result_df_CTSB.loc[:,['batch', 'prop']]
result_df_CTSB.columns = ['batch', 'prop_CTSB']

In [ ]:
result_df_CTSL = result_df[result_df['gene'] == 'CTSL'].copy()
result_df_CTSL = result_df_CTSL.reset_index(drop = True)
result_df_CTSL = result_df_CTSL.loc[:,['batch', 'prop']]
result_df_CTSL.columns = ['batch', 'prop_CTSL']

In [ ]:
results_df_raw = pd.merge(results_df_raw, result_df_CTSB, on = 'batch')

In [ ]:
results_df_raw = pd.merge(results_df_raw, result_df_CTSL, on = 'batch')

In [ ]:
results_df_raw.to_csv('../data/results_df_spatial_vessel_linkerenz.csv')